# Ordered Logistic Regression Results: Adoption Predictors Data Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset summary
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"License: {metadata.license}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, their fields, and IDs.

In [ ]:
# List all available record sets by @id
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets found in this dataset. Please check the Croissant schema or data documentation.')
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        # List field @id for each record set
        if 'field' in rs:
            if isinstance(rs['field'], dict):
                fields = [rs['field']]
            else:
                fields = rs['field']
            for fld in fields:
                print(f"    Field: {fld['@id']}")
        else:
            print('    No fields found.')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using record set and field `@id`s from the overview.

In [ ]:
# Gather all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets] if hasattr(dataset, 'record_sets') else []

dataframes = {}

if not record_set_ids:
    print("No record sets are defined in the dataset.")
else:
    for record_set_id in record_set_ids:
        # Extract records for each record set by @id
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set: {record_set_id}")
        except Exception as e:
            print(f"Could not load records for {record_set_id} due to: {e}")

    # Display columns of the first record set (if any)
    if record_set_ids:
        first_rs = record_set_ids[0]
        if first_rs in dataframes:
            print(f"\nColumns in record set {first_rs}:")
            print(dataframes[first_rs].columns.tolist())
            print(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some standard processing: filtering and normalization. We'll choose the first available numeric field from the first record set for demonstration.

In [ ]:
# EDA: Filter numeric field and normalize
import numpy as np

if not dataframes:
    print("No DataFrames loaded. Please ensure valid record sets exist in dataset.")
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to find a numeric field by inspecting dtypes
    numeric_field_id = None
    for col in df.columns:
        try:
            # Attempt to coerce to numeric
            col_numeric = pd.to_numeric(df[col], errors='coerce')
            if np.sum(~col_numeric.isnull()) > 0:
                numeric_field_id = col
                break
        except Exception:
            continue

    if not numeric_field_id:
        print("No numeric fields found for EDA in the first record set.")
    else:
        # Filter records with value > threshold
        threshold = 10
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')  # ensure numeric
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a non-numeric field if available
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("\nNo suitable non-numeric group field found for grouping in this record set.")

## 5. Visualization
Visualize the distribution of the chosen numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_field_id:
    print("Cannot display plot: no suitable numeric field available.")
else:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to ingest a Croissant FAIR^2 dataset using `mlcroissant` by referencing all entities with their unique `@id` fields, and how to perform a basic exploratory data analysis on the loaded data. This process helps ensure reproducibility and semantic clarity when working with rich, machine-readable data packages.